# 04c — ASR Transcription for the Text-Only Baseline

| Field | Detail |
|---|---|
| **Notebook** | `04c_asr_transcription.ipynb` |
| **Pipeline stage** | H1 — Voice Emotion Recognition - 4 of 7 (Feature Extraction, text branch) |
| **Owner(s)** | *Thrithwaka Preethi Shakya* |
| **Created** | *28/08/2026* |
| **Last updated** | *28/08/2026* |
| **Upstream dependency** | `02_preprocessing.ipynb` (manifest). Independent of `04a`/`04b` — reads only the manifest, not their feature outputs. |
| **Downstream dependency** | `05e_model_distilroberta.ipynb` |
| **Research proposal reference** | Section 5.1 (*"the voice-based model's classification accuracy will be directly benchmarked against a text-only emotion classifier (DistilRoBERTa on transcribed text) on the same evaluation set"*) |

## Purpose

H1 tests whether voice-based emotion recognition outperforms text-only emotion recognition. To run that comparison fairly, the text-only model needs to see **real transcripts of what was actually said** — not hand-typed ground-truth text, which would be an unrealistically clean input no real deployed system would have. This notebook produces those transcripts using the same ASR technology (`src/asr/speech_to_text.py`, Groq-hosted Whisper) already scaffolded elsewhere in this project, so the text-only baseline reflects a realistic, imperfect transcription pipeline — exactly what H1's comparison is supposed to measure.

### An important, deliberate difference from `04a` and `04b`

Both `04a` and `04b` read audio from `standardized_path` — the 2.5-second, 16kHz version produced by `02_preprocessing.ipynb`. **This notebook does not.** It transcribes from `original_path` instead, for a specific reason: the 2.5-second standardization exists only because the acoustic models (CNN, CNN-LSTM, Wav2Vec2) need fixed-length input tensors for batching. A transcript has no such requirement — DistilRoBERTa's tokenizer handles variable-length text natively. Since `03_eda.ipynb`'s duration profiling showed some clips (particularly in RAVDESS) run longer than 2.5 seconds, transcribing the *truncated* version risks cutting a sentence off mid-word and producing an incomplete, misleading transcript. Transcribing the full original recording avoids that failure mode entirely, at no cost, since nothing downstream of this notebook needs fixed-length audio.

## Objectives

1. Load and validate the manifest from `02_preprocessing.ipynb`.
2. Confirm ASR API credentials are configured before starting a batch job.
3. Transcribe every file's **original** (non-truncated) audio using the project's existing `src/asr/speech_to_text.py` module.
4. Implement checkpointed, resumable batch processing — this notebook may call an external, rate-limited API thousands of times, and must not lose all progress to a single dropped connection or an interrupted Colab session.
5. Implement a retry-with-backoff strategy for transient failures, and clearly log (not silently discard) any file that ultimately fails.
6. Diagnose transcript quality: empty-transcript rate, unusually short transcripts, and a manual spot-check sample.
7. Save the final transcript dataset, aligned to the manifest, plus full metadata.
8. Leave a structured handoff note for `05e_model_distilroberta.ipynb`.

## 0. Environment Setup

In [1]:
import sys
import json
import time
import logging
from pathlib import Path

import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import settings  # noqa: E402
from src.asr.speech_to_text import transcribe_audio  # noqa: E402 — reuses the project's existing ASR module

print(f"Project root resolved to: {PROJECT_ROOT}")

Project root resolved to: C:\Users\thrit\Desktop\emotion-ai-companion-research


In [2]:
LOG_DIR = PROJECT_ROOT / "reports" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / "04c_asr_transcription.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, mode="a"), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("asr_transcription")
# Note: log mode is 'a' (append), not 'w' — this notebook is resumable across multiple runs,
# and we want the log history to reflect that, unlike the other notebooks' single-shot runs.
RANDOM_SEED = 42

## 1. Load & Validate Upstream Outputs, and API Credentials

Before spending any API quota, this section fails fast and clearly if either the upstream manifest isn't ready or the Groq API key isn't configured — better to catch a missing `.env` entry in one line here than three minutes into a batch job.

In [3]:
preprocessing_report_path = PROJECT_ROOT / "reports" / "02_preprocessing_report.json"
if not preprocessing_report_path.exists():
    raise FileNotFoundError(f"{preprocessing_report_path} not found. Run 02_preprocessing.ipynb first.")

with open(preprocessing_report_path) as f:
    preprocessing_report = json.load(f)

assert preprocessing_report["class_count_validation_passed"], "Notebook 02 validation failed — fix before proceeding."
assert preprocessing_report["checksum_verification_passed"], "Notebook 02 checksum verification failed — fix before proceeding."

if not settings.groq_api_key:
    raise RuntimeError(
        "settings.groq_api_key is empty. Copy .env.example to .env, add a free Groq API key "
        "(https://console.groq.com), and restart the kernel before running this notebook."
    )

logger.info("Upstream validated. Groq API key is configured.")
print("\u2705 Upstream validated. Groq API key found.")

2026-08-30 05:58:03,938 | INFO | Upstream validated. Groq API key is configured.
✅ Upstream validated. Groq API key found.


In [4]:
manifest_path = PROJECT_ROOT / preprocessing_report["manifest_path"]
manifest_df = pd.read_csv(manifest_path)
for col in ["original_path", "standardized_path"]:
    manifest_df[col] = manifest_df[col].str.replace("\\\\", "/", regex=True)

handcrafted_metadata_path = PROJECT_ROOT / "data" / "processed" / "features" / "h1_handcrafted_features_metadata.json"
if handcrafted_metadata_path.exists():
    with open(handcrafted_metadata_path) as f:
        label_to_idx = {k: int(v) for k, v in json.load(f)["label_to_idx"].items()}
else:
    label_to_idx = {label: idx for idx, label in enumerate(settings.emotion_labels)}

print(f"Loaded manifest: {len(manifest_df)} rows")

Loaded manifest: 4068 rows


## 2. Configuration

**`SAMPLE_SIZE`** exists specifically so the whole pipeline — retry logic, checkpointing, output format — can be validated on a small, cheap subset before committing to transcribing the full ~4,000-file dataset. Set it to a small integer for a first test run, then to `None` for the full run. This is standard practice before running any large, costly, or rate-limited batch job, and costs nothing to include.

In [5]:
SAMPLE_SIZE = None          # set to None for the full dataset once the pipeline is validated
LANGUAGE = "en"           # all three source datasets (RAVDESS, TESS, SAVEE) are English
CHECKPOINT_EVERY_N = 50   # save progress to disk this often — protects against losing work to a
                          # dropped connection, a Colab timeout, or a rate-limit lockout mid-run
MAX_RETRIES = 3
RETRY_BACKOFF_BASE_SECONDS = 2  # exponential backoff: 2s, 4s, 8s between retries

FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = FEATURES_DIR / "h1_transcripts.csv"
OUTPUT_METADATA_PATH = FEATURES_DIR / "h1_transcripts_metadata.json"

work_df = manifest_df if SAMPLE_SIZE is None else manifest_df.sample(SAMPLE_SIZE, random_state=RANDOM_SEED)
print(f"This run will process {len(work_df)} file(s). SAMPLE_SIZE={SAMPLE_SIZE}")
if SAMPLE_SIZE is not None:
    print("\u26a0\ufe0f  Running on a SAMPLE, not the full dataset. Set SAMPLE_SIZE = None before the real run.")

This run will process 4068 file(s). SAMPLE_SIZE=None


## 3. Resume-from-Checkpoint Logic

If `h1_transcripts.csv` already exists from a previous (possibly interrupted) run of this notebook, already-transcribed files are loaded and skipped, rather than re-transcribed — saving both time and API quota. This makes the notebook safe to re-run after any interruption without losing prior progress or wasting a second round of API calls on files that already succeeded.

In [6]:
if OUTPUT_CSV_PATH.exists():
    existing_results_df = pd.read_csv(OUTPUT_CSV_PATH)
    already_done = set(existing_results_df["original_path"])
    logger.info("Found existing checkpoint with %d transcribed files. Resuming.", len(already_done))
    print(f"Resuming from checkpoint: {len(already_done)} files already transcribed.")
else:
    existing_results_df = pd.DataFrame(
        columns=["original_path", "transcript", "attempts", "succeeded"]
    )
    already_done = set()
    print("No existing checkpoint found — starting fresh.")

remaining_df = work_df[~work_df["original_path"].isin(already_done)]
print(f"Files remaining to transcribe this run: {len(remaining_df)}")

2026-08-30 05:58:15,573 | INFO | Found existing checkpoint with 3457 transcribed files. Resuming.
Resuming from checkpoint: 3457 files already transcribed.
Files remaining to transcribe this run: 611


## 4. Retry Wrapper

`src/asr/speech_to_text.py::transcribe_audio()` already catches exceptions internally and returns an empty string on failure — this keeps the shared module simple for its other use (live inference in the Streamlit app), but means this notebook cannot distinguish *"the API call failed"* from *"the clip genuinely contains no discernible speech"* purely from its return value. Rather than modify a shared module to fit one notebook's needs, this section wraps it with its own retry logic: an empty result triggers a retry (up to `MAX_RETRIES`, with exponential backoff), and if it's still empty after all retries, it's logged and accepted as the final result — most plausibly genuine silence rather than a persistent transient error. This is a deliberate, documented trade-off, not an oversight.

In [7]:
def transcribe_with_retry(audio_path: Path, language: str = LANGUAGE, max_retries: int = MAX_RETRIES) -> tuple[str, int]:
    """Returns (transcript, attempts_used). Retries on an empty result with exponential backoff."""
    for attempt in range(1, max_retries + 1):
        transcript = transcribe_audio(str(audio_path), language=language)
        if transcript.strip():
            return transcript.strip(), attempt
        if attempt < max_retries:
            backoff = RETRY_BACKOFF_BASE_SECONDS * (2 ** (attempt - 1))
            logger.warning("Empty transcript for %s (attempt %d/%d) — retrying in %ds.",
                           audio_path.name, attempt, max_retries, backoff)
            time.sleep(backoff)
    logger.warning("Empty transcript for %s after %d attempts — accepting as likely genuine silence.",
                   audio_path.name, max_retries)
    return "", max_retries

## 5. Batch Transcription (Checkpointed)

Processes the remaining files, saving progress to disk every `CHECKPOINT_EVERY_N` files. If this cell is interrupted for any reason, simply re-run the notebook from the top — Section 3 will pick up exactly where it left off.

In [8]:
new_results = []

for i, (_, row) in enumerate(tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Transcribing")):
    audio_path = PROJECT_ROOT / row["original_path"]  # deliberately original_path, not standardized_path — see Purpose above
    transcript, attempts = transcribe_with_retry(audio_path)

    new_results.append({
        "original_path": row["original_path"],
        "transcript": transcript,
        "attempts": attempts,
        "succeeded": bool(transcript.strip()),
    })

    if (i + 1) % CHECKPOINT_EVERY_N == 0 or (i + 1) == len(remaining_df):
        checkpoint_df = pd.concat([existing_results_df, pd.DataFrame(new_results)], ignore_index=True)
        checkpoint_df.to_csv(OUTPUT_CSV_PATH, index=False)
        logger.info("Checkpoint saved: %d / %d files transcribed this run.", i + 1, len(remaining_df))

print(f"\nTranscription run complete. {len(new_results)} file(s) processed this run.")

Transcribing:   0%|          | 0/611 [00:00<?, ?it/s]

2026-08-30 05:58:29,739 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-08-30 05:58:30,137 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-08-30 05:58:30,522 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-08-30 05:58:31,056 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-08-30 05:58:31,552 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-08-30 05:58:32,882 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-08-30 05:58:33,406 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-08-30 05:58:34,557 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/audio/transcriptions "HTTP/1.1 200 OK"
2026-08-30 05:58:36,897 

In [9]:
all_results_df = pd.concat([existing_results_df, pd.DataFrame(new_results)], ignore_index=True)
all_results_df = all_results_df.drop_duplicates(subset="original_path", keep="last").reset_index(drop=True)
all_results_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Total transcripts saved so far: {len(all_results_df)} / {len(manifest_df)} manifest rows")
if SAMPLE_SIZE is not None:
    print("\u26a0\ufe0f  This was a SAMPLE run. Set SAMPLE_SIZE = None and re-run from the top to process the full dataset.")

Total transcripts saved so far: 4068 / 4068 manifest rows


## 6. Merge With Manifest and Compute Quality Metrics

Joins the transcripts back onto the full manifest (label, split, source dataset) for quality analysis and for the final saved artifact.

In [10]:
# Merge manifest metadata with transcription results
merged_df = manifest_df.merge(
    all_results_df,
    on="original_path",
    how="inner",
    suffixes=("", "_result")
)

# Create transcript statistics
merged_df["transcript"] = merged_df["transcript"].fillna("")

merged_df["transcript_word_count"] = (
    merged_df["transcript"]
    .str.split()
    .str.len()
)

merged_df["transcript_char_count"] = (
    merged_df["transcript"]
    .str.len()
)

# Make sure succeeded exists
if "succeeded" not in merged_df.columns:
    merged_df["succeeded"] = (
        merged_df["transcript"].str.strip().str.len() > 0
    )

# Display summary
print(f"Merged dataset: {len(merged_df)} rows")

# Select only columns that actually exist
display_columns = [
    col for col in [
        "source_dataset",
        "label",
        "transcript",
        "transcript_word_count",
        "succeeded"
    ]
    if col in merged_df.columns
]

display(merged_df[display_columns].head(10))

Merged dataset: 4068 rows


,source_dataset,label,transcript,transcript_word_count,succeeded
0,ravdess,neutral,Dogs are sitting by the door.,6,True
1,tess,happy,Say the word date.,4,True
2,tess,sad,Say the word rag.,4,True
3,tess,happy,Say the word hush.,4,True
4,tess,sad,Say the word later.,4,True
5,ravdess,surprise,Kids are talking by the door?,6,True
6,tess,neutral,Say the word rag.,4,True
7,tess,fear,Say the word moon.,4,True
8,savee,fear,Will Robin wear a yellow,5,True
9,tess,surprise,Say the word higher.,4,True


In [11]:
print("Transcription success rate by source dataset:")
display(merged_df.groupby("source_dataset")["succeeded"].agg(["mean", "sum", "count"]).rename(
    columns={"mean": "success_rate", "sum": "succeeded_count", "count": "total"}
))

print("\nTranscript word count distribution by source dataset:")
display(merged_df.groupby("source_dataset")["transcript_word_count"].describe()[["min", "mean", "50%", "max"]])

empty_count = int((~merged_df["succeeded"]).sum())
very_short_count = int((merged_df["transcript_word_count"] <= 1).sum())
print(f"\nEmpty/failed transcripts: {empty_count} ({empty_count / len(merged_df):.1%})")
print(f"Very short transcripts (\u22641 word): {very_short_count} ({very_short_count / len(merged_df):.1%})")

Transcription success rate by source dataset:


,success_rate,succeeded_count,total
source_dataset,,,
ravdess,1.000000,1248,1248
savee,1.000000,420,420
tess,0.999167,2398,2400



Transcript word count distribution by source dataset:


,min,mean,50%,max
source_dataset,,,,
ravdess,5.0,6.019231,6.0,7.0
savee,2.0,8.819048,8.0,17.0
tess,0.0,3.998750,4.0,5.0



Empty/failed transcripts: 2 (0.0%)
Very short transcripts (≤1 word): 2 (0.0%)


### Manual spot-check

Automated metrics can't catch every quality issue — a transcript can have the "right" word count and still be wrong. This prints a handful of real examples for a human to read and judge, paired with the file's known emotion label, so obviously wrong transcriptions (e.g. a transcript that's clearly a different sentence than what the dataset's actors were scripted to say) can be caught before this data trains a model.

In [12]:
sample_check = merged_df.sample(min(10, len(merged_df)), random_state=RANDOM_SEED)
for _, row in sample_check.iterrows():
    print(f"[{row['source_dataset'].upper()} | {row['label']}] \"{row['transcript']}\"")

[TESS | neutral] "Say the word, ripe."
[SAVEE | fear] ""'Will Robin wear a yellow lily?'"
[SAVEE | neutral] "The carpet cleaners shampooed our oriental rug."
[RAVDESS | angry] "Kids are talking by the door."
[TESS | happy] "Say the word kill."
[RAVDESS | surprise] "Kids are talking by the door."
[TESS | happy] "Say the word dip."
[RAVDESS | angry] "Kids are talking by the door."
[SAVEE | neutral] "As such, it was beyond politics and had no need of justification by a message."
[TESS | surprise] "Say the word room."


## 7. Handling Empty Transcripts — A Decision, Not a Silent Default

Some clips will legitimately fail to transcribe — very short exclamatory utterances, or emotionally distorted speech (e.g. an extreme angry or fearful reading) can genuinely defeat ASR. An empty transcript carries zero information for a text classifier, and including it as a training example would just teach the model to associate "no text" with whatever emotion happened to be attached, which is meaningless. **These rows are flagged, not silently dropped here** — the decision to exclude them belongs to `05e_model_distilroberta.ipynb`, which should filter on the `succeeded` column at training time, using this notebook's saved data as the single source of truth for *why* each row was included or excluded.

In [13]:
print("Rows flagged as empty/failed, by source dataset and label (for the team to review):")

succeeded = merged_df["succeeded"]

if succeeded.dtype == bool:
    failed_rows = merged_df[~succeeded]
else:
    failed_rows = merged_df[succeeded == 0]

display(
    failed_rows
    .groupby(["source_dataset", "label"])
    .size()
    .rename("count")
    .reset_index()
)

Rows flagged as empty/failed, by source dataset and label (for the team to review):


,source_dataset,label,count
0,tess,angry,1
1,tess,fear,1


## 8. Save Final Transcript Dataset and Metadata

In [14]:
final_columns = [
    "original_path", "standardized_path", "source_dataset", "speaker_id", "gender",
    "label", "split", "transcript", "transcript_word_count", "transcript_char_count",
    "attempts", "succeeded",
]
merged_df[final_columns].to_csv(OUTPUT_CSV_PATH, index=False)
logger.info("Final transcript dataset saved to %s (%d rows)", OUTPUT_CSV_PATH, len(merged_df))

transcript_metadata = {
    "random_seed": RANDOM_SEED,
    "asr_model": "whisper-large-v3 (via Groq API)",
    "language": LANGUAGE,
    "audio_source": "original_path (full, non-truncated duration) — see Purpose section for rationale",
    "max_retries": MAX_RETRIES,
    "total_manifest_rows": int(len(manifest_df)),
    "total_transcribed_rows": int(len(merged_df)),
    "succeeded_count": int(merged_df["succeeded"].sum()),
    "empty_or_failed_count": int((~merged_df["succeeded"]).sum()),
    "empty_or_failed_rate": float((~merged_df["succeeded"]).mean()),
    "very_short_transcript_count": int(very_short_count),
    "label_to_idx": label_to_idx,
    "output_csv_path": str(OUTPUT_CSV_PATH.relative_to(PROJECT_ROOT)),
    "recommendation_for_05e": "Filter training data on succeeded == True before fine-tuning DistilRoBERTa.",
}

with open(OUTPUT_METADATA_PATH, "w") as f:
    json.dump(transcript_metadata, f, indent=2)

logger.info("Saved transcript metadata to %s", OUTPUT_METADATA_PATH)
print(f"Saved: {OUTPUT_CSV_PATH}")
print(f"Saved: {OUTPUT_METADATA_PATH}")
print(json.dumps(transcript_metadata, indent=2))

2026-08-30 06:43:46,495 | INFO | Final transcript dataset saved to C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_transcripts.csv (4068 rows)
2026-08-30 06:43:46,503 | INFO | Saved transcript metadata to C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_transcripts_metadata.json
Saved: C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_transcripts.csv
Saved: C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_transcripts_metadata.json
{
  "random_seed": 42,
  "asr_model": "whisper-large-v3 (via Groq API)",
  "language": "en",
  "audio_source": "original_path (full, non-truncated duration) \u2014 see Purpose section for rationale",
  "max_retries": 3,
  "total_manifest_rows": 4068,
  "total_transcribed_rows": 4068,
  "succeeded_count": 4066,
  "empty_or_failed_count": 2,
  "empty_or_failed_rate": 0.0004916420845624386,
  "very_short_transcript_count": 2,
  "label_to_id

## 9. Final Sanity Checks

In [15]:
reloaded_df = pd.read_csv(OUTPUT_CSV_PATH)

assert len(reloaded_df) == len(merged_df), "Row count mismatch after reload!"
assert reloaded_df["original_path"].is_unique, "Duplicate original_path values found — alignment risk!"
assert set(reloaded_df["label"].unique()).issubset(set(settings.emotion_labels)), "Unexpected label found!"

if SAMPLE_SIZE is None:
    coverage = len(reloaded_df) / len(manifest_df)
    print(f"Coverage: {coverage:.1%} of the full manifest has a transcript on file.")
    if coverage < 1.0:
        print("\u26a0\ufe0f  Not all manifest rows are covered — re-run this notebook to process the remainder.")
else:
    print(f"\u26a0\ufe0f  SAMPLE_SIZE={SAMPLE_SIZE} — this was a test run, not full coverage. This is expected.")

print("\n\u2705 No duplicate original_path values (safe to join against the manifest downstream).")
print("\u2705 All labels within the expected 6-class scheme.")

Coverage: 100.0% of the full manifest has a transcript on file.

✅ No duplicate original_path values (safe to join against the manifest downstream).
✅ All labels within the expected 6-class scheme.


---

## Summary Note — Handoff to Next Notebook

### What was done in this notebook

1. Validated `02_preprocessing.ipynb`'s report and confirmed the Groq API key is configured before starting.
2. Transcribed every file's **original, non-truncated** audio (not the 2.5s-standardized version used by `04a`/`04b`) via the project's existing `src/asr/speech_to_text.py` module, using `language="en"`.
3. Implemented checkpointed, resumable batch processing (saves every 50 files) and a retry-with-backoff wrapper for transient failures.
4. Diagnosed transcript quality: success rate, word-count distribution, and a manual spot-check sample, broken down by source dataset.
5. Explicitly flagged (not silently dropped) empty/failed transcripts, deferring the exclusion decision to `05e_model_distilroberta.ipynb`.
6. Saved the final transcript dataset (aligned to the manifest) and full metadata.
7. Verified the saved file has no duplicate keys and full or expected partial coverage of the manifest.

### Outputs produced by this notebook (and where to find them)

| Output | Location | Used by |
|---|---|---|
| Transcript dataset | `data/processed/features/h1_transcripts.csv` | `05e_model_distilroberta.ipynb` — the direct training data source |
| Transcript metadata | `data/processed/features/h1_transcripts_metadata.json` | Confirms coverage, empty-rate, and the filtering recommendation for `05e` |
| Run log (append mode — accumulates across resumed runs) | `reports/logs/04c_asr_transcription.log` | Debugging retries, rate-limit events, or resumption history |

### How to load this notebook's output (for `05e`)

```python
import pandas as pd
import json

transcripts_df = pd.read_csv("data/processed/features/h1_transcripts.csv")
with open("data/processed/features/h1_transcripts_metadata.json") as f:
    metadata = json.load(f)

# Recommended: filter out empty/failed transcripts before training
training_df = transcripts_df[transcripts_df["succeeded"]]

train_texts = training_df.loc[training_df["split"] == "train", "transcript"]
train_labels = training_df.loc[training_df["split"] == "train", "label"].map(metadata["label_to_idx"])
```

### What needs to be done next

With `04a`, `04b`, and `04c` all complete, **all four H1 candidate voice/text inputs are now ready**. Proceed to model training:

- **`05a_model_cnn_raw.ipynb`** — reads `04b`'s `X_raw_cnn`
- **`05b_model_cnn_lstm.ipynb`** — reads `04a`'s `X_set_a`
- **`05c_model_ensemble.ipynb`** — reads `04a`'s `X_set_b`
- **`05d_model_wav2vec2.ipynb`** — reads `04b`'s `X_wav2vec2`
- **`05e_model_distilroberta.ipynb`** — reads this notebook's `h1_transcripts.csv`

**Before running `05e`, confirm:**
- This notebook was run with `SAMPLE_SIZE = None` (a full run, not a test sample).
- Section 9's coverage check reports 100% (or explain in Known Issues below why it doesn't).
- `h1_transcripts_metadata.json`'s `empty_or_failed_rate` is a value the team has actually looked at and decided is acceptable — don't proceed on autopilot if it's unexpectedly high (e.g. above ~10%), since that would suggest a systematic ASR problem (wrong language setting, audio path bug) rather than genuine data-level silence.

### Resources needed for the next step

| Resource | Needed for | Notes |
|---|---|---|
| `data/processed/features/h1_transcripts.csv` | Direct training data for `05e` | Filter on `succeeded == True` before use, per the recommendation in Section 7 |
| `data/processed/features/h1_transcripts_metadata.json` | Label mapping consistency check, filtering recommendation | Same `label_to_idx` as every other H1 notebook |
| A Hugging Face `distilroberta-base` checkpoint | Fine-tuning in `05e` | Downloaded automatically by `transformers` on first use — needs network access once |

### Known issues / things to watch for

- *None*

### Run metadata

- **Run by:** *Thrithwaka Preethi Shakya*
- **Date:** *29/08/2026*
- **Full run (`SAMPLE_SIZE = None`) completed:** *Yes*
- **Final coverage:** *100%*
- **Empty/failed transcript rate:** *0.1%*